In [1]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool


In [2]:
@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search results for: {query}"


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {to} with subject: {subject}"


@tool
def delete_records(table: str, condition: str) -> str:
    """Delete records from the database."""
    return f"Deleted records from {table} where {condition}"

In [3]:
checkpointer = InMemorySaver()

In [18]:
# Step 1: Invoke - agent will pause before send_email
config = {"configurable": {"thread_id": "session_001"}}

result = hitl_agent.invoke(
{"messages": [{"role": "user", "content": "Send an email to team@company.com about the Q4 results"}]},
config=config

)

print(" === Agent paused - awaiting human approval === ")
print(result)

 === Agent paused - awaiting human approval === 
{'messages': [HumanMessage(content='Send an email to team@company.com about the Q4 results', additional_kwargs={}, response_metadata={}, id='b188cf76-7014-44e7-9859-9f312db155e1'), AIMessage(content='Sure, I can help with that. Could you please provide the subject and the body of the email?', additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 221, 'total_tokens': 243, 'completion_tokens': 22, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-small-2506', 'model': 'mistral-small-2506', 'finish_reason': 'stop', 'model_provider': 'mistralai'}, id='lc_run--019f65f4-7f9a-7371-beba-394f438e840e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 221, 'output_tokens': 22, 'total_tokens': 243})]}


In [19]:
# Step 1: Invoke - agent will pause before send_email
config = {"configurable": {"thread_id": "session_001"}}

result = hitl_agent.invoke(
{"messages": [{"role": "user", "content": "Send an email to team@company.com about the Q4 results"}]},
config=config

)

print(" === Agent paused - awaiting human approval === ")
print(result)

 === Agent paused - awaiting human approval === 
{'messages': [HumanMessage(content='Send an email to team@company.com about the Q4 results', additional_kwargs={}, response_metadata={}, id='b188cf76-7014-44e7-9859-9f312db155e1'), AIMessage(content='Sure, I can help with that. Could you please provide the subject and the body of the email?', additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 221, 'total_tokens': 243, 'completion_tokens': 22, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-small-2506', 'model': 'mistral-small-2506', 'finish_reason': 'stop', 'model_provider': 'mistralai'}, id='lc_run--019f65f4-7f9a-7371-beba-394f438e840e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 221, 'output_tokens': 22, 'total_tokens': 243}), HumanMessage(content='Send an email to team@company.com about the Q4 results', additional_kwargs={}, response_metadata={}, id='68cdcaf0-965f-481c-b4b2-25ea4ccd988c'), AIMessage(content='Su

In [20]:
# Step 2: Human reviews and APPROVE
approved_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type" : "approve"}]}),
    config=config     # Same thread_id resumes the paused session
)

print(" === Approved! Final response === ")
print(approved_result["messages"][-1].content)

 === Approved! Final response === 
Sure, but I need the subject and body of the email.


In [21]:
# Step 3: Alternative - Human REJECTS
config2 = {"configurable": {"thread_id": "session_002"}}

hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Delete all records from the users table where active=false"}]},
    config=config2

)

rejected_result = hitl_agent. invoke(
    Command(resume={"decisions": [{"type" : "reject","peason": "Too risky, needs DBA review"}]}),
    config=config2
)

print(" === Rejected! Final response === ")
print(rejected_result["messages"][-1].content)

 === Rejected! Final response === 
To delete all records from the users table where active is false, I will execute the following SQL command:

```sql
DELETE FROM users WHERE active = false;
```

Would you like to proceed with this deletion?
